# Ask Spurgeon - Train on 1500 Examples (Colab Ready)

**Goal**: Fine-tune Llama-3.1-8B on the prepared 1500 high-fidelity Spurgeon examples using QLoRA + Unsloth.

This notebook is optimized for **Google Colab free T4** (or Pro A100).

### Quick Start
1. Runtime → Change runtime type → GPU (T4 recommended)
2. Run all cells
3. Training should take 3–8 hours on T4 (depending on settings)

Dataset: 1500 examples from `spurgeon_train_1500.jsonl`

In [1]:
# @title 1. Install Dependencies (Run this first)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps xformers "trl" peft accelerate bitsandbytes -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# @title Hugging Face Authentication (Required for gated Llama-3.1 model)
import os
from pathlib import Path

# 1. Try to load from Google Colab Secrets first
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ["HF_TOKEN"] = token
        print("✅ HF_TOKEN loaded from Google Colab Secrets!")
except Exception:
    pass

# 2. Try to load from local .env if running in a local environment
if not os.environ.get("HF_TOKEN"):
    for env_path in [Path(".env"), Path("../.env"), Path("../../.env"), Path("ask-spurgeon/.env"), Path("/content/ask-spurgeon/.env")]:
        if env_path.exists():
            with open(env_path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip().startswith("HF_TOKEN="):
                        token = line.strip().split("=", 1)[1].strip("'\" ")
                        if token:
                            os.environ["HF_TOKEN"] = token
                            print(f"✅ HF_TOKEN loaded from local file: {env_path}")
                            break

# 3. If still not authenticated, prompt interactive login
if not os.environ.get("HF_TOKEN"):
    print("🔑 HF_TOKEN not found in Secrets or .env. Prompting interactive login...")
    try:
        from huggingface_hub import notebook_login
        notebook_login()
    except ImportError:
        print("huggingface_hub not installed. Run pip install huggingface_hub")
else:
    print("🎉 Successfully authenticated with Hugging Face!")

In [2]:
# @title 2. Clone the repository
!git clone https://github.com/elViRafa/ask-spurgeon.git
%cd ask-spurgeon

fatal: destination path 'ask-spurgeon' already exists and is not an empty directory.
/content/ask-spurgeon


In [3]:
# @title 3. (Optional) Mount Google Drive to save model
from google.colab import drive
drive.mount('/content/drive')

# Change this if you want to save to Drive
SAVE_PATH = "/content/drive/MyDrive/Spurgeon_8B_QLoRA_1500"
print("Models will be saved to:", SAVE_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Models will be saved to: /content/drive/MyDrive/Spurgeon_8B_QLoRA_1500


In [4]:
# @title 4. Training Configuration (Good for T4)
MODEL_NAME = "unsloth/llama-3.1-8b-instruct-bnb-4bit"
DATASET_PATH = "fine_tuning/data/spurgeon_train_1500.jsonl"
OUTPUT_DIR = SAVE_PATH if 'SAVE_PATH' in globals() else "outputs/spurgeon-8b-1500"

# Conservative settings for free T4 (16GB VRAM)
MAX_SEQ_LENGTH = 2048          # Reduce if you get OOM
BATCH_SIZE = 1
GRAD_ACCUM = 16                # Effective batch = 16
LEARNING_RATE = 1e-4
EPOCHS = 1                     # Start with 1 epoch for testing
LORA_R = 32                    # Lower rank for T4

print("Configuration ready for 1500 examples.")

Configuration ready for 1500 examples.


In [5]:
# @title 5. Load Model + Apply LoRA (Unsloth)
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print("Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    lora_alpha = 64,
    lora_dropout = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
)

tokenizer = get_chat_template(tokenizer, chat_template="llama-3")
print("Model loaded and LoRA applied!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model...
==((====))==  Unsloth 2026.5.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.9 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Model loaded and LoRA applied!


In [6]:
# @title 6. Load Dataset
from datasets import load_dataset
from pathlib import Path

dataset_path = Path(DATASET_PATH)

if not dataset_path.exists():
    print("Dataset not found in the repo.")
    print("Please upload 'spurgeon_train_1500.jsonl' using the Colab file browser (left sidebar → Files → Upload).")
    print("Or mount Google Drive and set the correct path.")
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}")

print(f"Loading dataset: {DATASET_PATH}")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

def formatting_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True, remove_columns=dataset.column_names)
print(f"Loaded {len(dataset)} examples")

Loading dataset: fine_tuning/data/spurgeon_train_1500.jsonl
Loaded 1500 examples


In [8]:
import torch
import inspect
import transformers
from transformers import Trainer

# Runtime Monkeypatch to fix transformers/trl version incompatibility
original_init = Trainer.__init__
def patched_init(self, *args, **kwargs):
    if "tokenizer" in kwargs:
        if "processing_class" not in kwargs:
            kwargs["processing_class"] = kwargs.pop("tokenizer")
        else:
            kwargs.pop("tokenizer")
    return original_init(self, *args, **kwargs)
Trainer.__init__ = patched_init
transformers.Trainer.__init__ = patched_init

try:
    import unsloth.models._utils as unsloth_utils
    if hasattr(unsloth_utils, "_original_trainer_init"):
        orig_unsloth_init = unsloth_utils._original_trainer_init
        def patched_unsloth_init(self, *args, **kwargs):
            if "tokenizer" in kwargs:
                if "processing_class" not in kwargs:
                    kwargs["processing_class"] = kwargs.pop("tokenizer")
                else:
                    kwargs.pop("tokenizer")
            return orig_unsloth_init(self, *args, **kwargs)
        unsloth_utils._original_trainer_init = patched_unsloth_init
except Exception as e:
    pass

# @title 7. Start Training
from trl import SFTTrainer
try:
    from trl import SFTConfig
except ImportError:
    SFTConfig = None

# Safer settings for T4 GPU (most free Colab users)
fp16 = not torch.cuda.is_bf16_supported()
bf16 = torch.cuda.is_bf16_supported()

if SFTConfig is not None:
    # Modern TRL API
    training_args = SFTConfig(
        output_dir = OUTPUT_DIR,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps = 20,
        num_train_epochs = EPOCHS,
        learning_rate = LEARNING_RATE,
        fp16 = fp16,
        bf16 = bf16,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        save_strategy = "epoch",
        max_seq_length = MAX_SEQ_LENGTH,
        dataset_text_field = "text",
        packing = False,
    )
    
    # Check signature for tokenizer/processing_class compatibility
    sig = inspect.signature(SFTTrainer.__init__)
    trainer_kwargs = {
        "model": model,
        "train_dataset": dataset,
        "args": training_args,
    }
    if "processing_class" in sig.parameters:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer
        
    trainer = SFTTrainer(**trainer_kwargs)
else:
    # Legacy TRL API
    from transformers import TrainingArguments
    training_args = TrainingArguments(
        output_dir = OUTPUT_DIR,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps = 20,
        num_train_epochs = EPOCHS,
        learning_rate = LEARNING_RATE,
        fp16 = fp16,
        bf16 = bf16,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        save_strategy = "epoch",
    )
    
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = dataset,
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        args = training_args,
        packing = False,
    )

print("Starting training on 1500 examples...")
trainer.train()

TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
# @title 8. Save the Model
print("Saving LoRA adapter...")
trainer.save_model(OUTPUT_DIR)
print(f"LoRA saved to: {OUTPUT_DIR}")

# Optional: Merge and save 16-bit version
# model.save_pretrained_merged(f"{OUTPUT_DIR}-merged", tokenizer, save_method="merged_16bit")
# print("Merged model saved!")

## 9. Test the Fine-Tuned Model (Inference)

Now that the model is trained, let's verify that it actually speaks in Spurgeon's voice while remaining **faithful** to the retrieved context.

We will load a test context and question, format it using Llama-3's chat template, and use Unsloth's fast inference with **real-time streaming** to generate the response.

In [ ]:
# @title 9. Real-Time Inference Test
from unsloth import FastLanguageModel
from transformers import TextStreamer

# 1. Set model to fast inference mode
FastLanguageModel.for_inference(model)

# 2. Prepare a test sermon context and a question
test_context = (
    "In my sermon 'Songs in the Night', I declared: 'Any man can sing in the day. "
    "It is easy to sing when we can read the notes by daylight; but the skillful singer "
    "is he who can sing when there is not a ray of light to read by. Songs in the night come "
    "only from God; they are not in the power of man. They are the sovereign gifts of heaven.'"
)
test_question = "Where do songs in the night come from, and is it easy for man to sing them?"

# 3. Format using Llama-3 chat template (matching the training format)
messages = [
    {
        "role": "system",
        "content": (
            "You are Charles Haddon Spurgeon. You must answer the user's question "
            "using ONLY the information in the CONTEXT section below. Do not add external knowledge.\n\n"
            f"CONTEXT:\n{test_context}"
        )
    },
    {
        "role": "user",
        "content": test_question
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

# 4. Setup text streamer for real-time output
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("=== Retrieved Context ===")
print(test_context)
print("\n=== User Question ===")
print(test_question)
print("\n=== Generated Answer (Spurgeon's Voice) ===")

# 5. Generate with streaming
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    use_cache = True,
    temperature = 0.6,
    top_p = 0.9,
)

## After Training

1. Download the adapter from the output folder (or Drive)
2. Upload to Hugging Face (optional)
3. Use it in your RAG pipeline as the generator

See `fine_tuning/FINE_TUNING_PLAN.md` for next steps and evaluation ideas.

## 10. Merge LoRA Adapter (for TGI Deployment)

After training, you usually only have a **LoRA adapter**. To deploy with TGI (or for easier use), it is highly recommended to **merge** the adapter weights into the base model.

This section helps you do that directly in Colab.

In [ ]:
# @title 10. Merge LoRA Adapter (Fast Unsloth Native Method)
from unsloth import FastLanguageModel
from pathlib import Path
import os

# --- CONFIGURATION ---
# Where your trained LoRA adapter is saved (e.g. Google Drive)
ADAPTER_PATH = "/content/drive/MyDrive/Spurgeon_8B_QLoRA_1500"

# Where to save the merged model
MERGED_SAVE_PATH = "/content/drive/MyDrive/Spurgeon-8B-Merged-16bit"

# 1. Check if model is already loaded in the active session
if 'model' in globals() and 'tokenizer' in globals():
    print("✅ Found trained model and tokenizer in the active session. Using them for merging.")
else:
    print(f"🔄 Model not found in session. Loading trained adapter from: {ADAPTER_PATH}")
    if not Path(ADAPTER_PATH).exists():
        print(f"❌ Error: Adapter path {ADAPTER_PATH} does not exist.")
        print("Please ensure your Google Drive is mounted and the path is correct!")
    else:
        # Load the base model and automatically attach the saved LoRA adapter
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = ADAPTER_PATH,
            max_seq_length = 2048, # Must match your training max_seq_length
            load_in_4bit = True,   # Low RAM loading
        )
        print("✅ Model and adapter loaded successfully from disk!")

# 2. Perform the optimized merge
if 'model' in globals():
    print(f"Merging adapter and saving 16-bit model to: {MERGED_SAVE_PATH}")
    print("This will download the base model optimized weights and merge them natively.")
    print("It is highly optimized and prevents Colab System RAM from crashing!")
    
    model.save_pretrained_merged(
        MERGED_SAVE_PATH,
        tokenizer,
        save_method="merged_16bit",
    )
    
    print("\n✅ Merge completed successfully via Unsloth!")
    print(f"Merged model saved at: {MERGED_SAVE_PATH}")
else:
    print("❌ Error: No model available to merge.")

## 11. Upload Merged Model to Hugging Face

Depending on where you merged your model, choose the appropriate option to upload it:

### Option A: Local PC Upload (Recommended if you merged locally on your PC CPU/GPU)
If you ran the merge locally on your computer, the model files are stored on your local hard drive. **Google Colab's cloud server cannot access your local C:\ drive directly.**

To upload the model, simply open a terminal **locally on your Windows PC** in the `search-sermons` directory and run:
```powershell
.venv\Scripts\python.exe fine_tuning\scripts\upload_to_hf.py
```
*(This standalone utility will read your Hugging Face write token from `.env` or prompt you interactively, and upload the local folder immediately!)*

### Option B: Cloud Colab Upload (If you merged using the Colab T4/A100 GPU)
If you successfully ran the merge inside the Google Colab cloud runtime (Step 10 above), the merged model is stored on your Google Drive.

1. Mount your Google Drive.
2. Ensure `MERGED_SAVE_PATH` in the cell below is set to your Google Drive path:
```python
MERGED_SAVE_PATH = "/content/drive/MyDrive/Spurgeon-8B-Merged-16bit"
```
3. Run the code cell below to perform the upload from the cloud container!

In [ ]:
# @title 11. (Optional) Upload Merged Model to Hugging Face
from huggingface_hub import login, HfApi
import os

# Use MERGED_SAVE_PATH from step 10 if available and exists, otherwise default to Drive path
if 'MERGED_SAVE_PATH' in globals() and os.path.exists(globals()['MERGED_SAVE_PATH']):
    upload_path = globals()['MERGED_SAVE_PATH']
else:
    upload_path = "/content/drive/MyDrive/Spurgeon-8B-Merged-16bit"

HF_REPO_ID = "rafaelvieirar1r/llama-3.1-8b-spurgeon-generator"   # <-- CHANGE THIS

print("Logging into Hugging Face...")
login()   # You will be prompted for your HF write token

print(f"Uploading merged model from: {upload_path} to repo: {HF_REPO_ID}")
api = HfApi()
api.upload_folder(
    folder_path=upload_path,
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Upload merged Spurgeon fine-tuned model (1500 examples)"
)

print("\n✅ Upload complete!")
print(f"Your model is now available at: https://huggingface.co/{HF_REPO_ID}")

Logging into Hugging Face...
Uploading merged model to: rafaelvieirar1r/llama-3.1-8b-spurgeon-generator


ValueError: Provided path: '/content/C:\Users\rafael\Projetos\search-sermons\fine_tuning\models\Spurgeon-8B-Merged-16bit' is not a directory

## Next Steps After Merging

1. Download the merged model from your Drive (or the path you chose).
2. Upload it to the Hugging Face Hub (the cell above can do this).
3. Use the TGI deployment guide in `fine_tuning/spaces/tgi-deployment/DEPLOY_TGI.md` to create a free inference Space.
4. Call your fine-tuned Spurgeon generator from your main RAG app using the OpenAI-compatible endpoint.

## 12. Convert Merged Model to GGUF (for CPU / llama.cpp Deployment)

If you want to deploy your fine-tuned model on **free CPU** (via Hugging Face Spaces or locally with Ollama/llama.cpp), you need to convert the merged model to **GGUF** format.

This is the recommended format for efficient CPU inference.

**Note**: This step is heavy and benefits greatly from a GPU runtime in Colab.

In [ ]:
# @title 12.1 Clone llama.cpp (needed for GGUF conversion)
!git clone https://github.com/ggerganov/llama.cpp --depth 1
%cd llama.cpp
!make -j
%cd ..

In [ ]:
# @title 12.2 Convert Merged Model to GGUF (f16 first)
from pathlib import Path

# Uses the MERGED_SAVE_PATH from the previous merge cell
MERGED_MODEL_PATH = MERGED_SAVE_PATH

GGUF_OUTPUT_DIR = "/content/drive/MyDrive/Spurgeon_GGUF"
Path(GGUF_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

OUTPUT_GGUF = f"{GGUF_OUTPUT_DIR}/Spurgeon-8B-f16.gguf"

print(f"Converting merged model to GGUF (f16)...")
!python llama.cpp/convert_hf_to_gguf.py \
    {MERGED_MODEL_PATH} \
    --outfile {OUTPUT_GGUF} \
    --outtype f16

print(f"\n✅ f16 GGUF created at: {OUTPUT_GGUF}")

In [ ]:
# @title 12.3 (Recommended) Quantize to Q4_K_M for CPU
# This produces a much smaller and faster model (~4.5-5 GB)

QUANTIZED_OUTPUT = f"{GGUF_OUTPUT_DIR}/Spurgeon-8B-Q4_K_M.gguf"

print("Quantizing to Q4_K_M...")
!llama.cpp/quantize {OUTPUT_GGUF} {QUANTIZED_OUTPUT} q4_k_m

print(f"\n✅ Quantized GGUF ready at: {QUANTIZED_OUTPUT}")
print("Upload this file for efficient CPU deployment.")

## After GGUF Conversion

1. Upload the quantized GGUF file (`Spurgeon-8B-Q4_K_M.gguf`) to a Hugging Face repo.
2. Use the ready CPU deployment files in `fine_tuning/spaces/cpu-llama-cpp/` to create a free inference Space.
3. Update your main RAG app to use the new endpoint (`LLM_PROVIDER=openai`).